# Libraries

In [1]:
import re
import json
import statistics
from pathlib import Path
from typing import List, Dict, Any

# We use the MiniLM tokenizer to count tokens precisely.
# Why this tokenizer?
# Because chunks will be embedded with MiniLM and BGE-Small.
# BGE-Small uses the same BPE tokenizer as BERT/MiniLM.
# Counting by word split() is inaccurate: "softmax" is 1 word but
# may be 2-3 BPE tokens ["soft", "##max"], silently exceeding model limits.
from transformers import AutoTokenizer

TOKENIZER = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

def count_tokens(text: str) -> int:
    """Count real BPE tokens, excluding special tokens ([CLS], [SEP])."""
    return len(TOKENIZER.encode(text, add_special_tokens=False))

print("Tokenizer loaded")
test = "The attention mechanism computes a weighted sum."
print(f"  Example: '{test}'")
print(f"  Words: {len(test.split())} | BPE tokens: {count_tokens(test)}")

c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run P

Tokenizer loaded
  Example: 'The attention mechanism computes a weighted sum.'
  Words: 7 | BPE tokens: 9


# Paths

In [2]:
BASE_DIR = Path("../")
SILVER   = BASE_DIR / "data" / "silver"    # clean texts
CHUNK     = BASE_DIR / "data" / "chunks"      # chunks (output of this notebook)
META_DIR = BASE_DIR / "data" / "metadata"

CHUNK.mkdir(parents=True, exist_ok=True)

txt_files = sorted(SILVER.glob("*.txt"))
print(f"✓ {len(txt_files)} texts found in {SILVER}")
for f in txt_files:
    print(f"  - {f.name}")

✓ 10 texts found in ..\data\silver
  - 1502.04681v3.txt
  - 1512.03385v1.txt
  - 1910.13461v1.txt
  - 2312.11514v3.txt
  - 2312.15872v1.txt
  - 2312.17482v2.txt
  - 2407.15516v1.txt
  - 2504.08386v1.txt
  - 2606.03748v1.txt
  - AttentionIsAllYouNeed.txt


# Equation Finding

In [3]:
GREEK    = set('αβγδεζηθλμνξπρστφψωΑΒΓΔΛΣΩ')
MATH_OPS = set('∑∫∂∇→←≈≠≤≥∈∉×÷·√')

def is_equation_line(line: str) -> bool:
    """
    Determine whether a text line is a math equation.
    Returns True if the line shows equation signals, False if it looks like prose.
    """
    stripped = line.strip()
    if not stripped or len(stripped) > 300:
        return False

    # --- Positive signals ---
    has_greek       = any(c in GREEK for c in stripped)
    math_op_count   = sum(1 for c in stripped if c in MATH_OPS)
    has_func_pattern = bool(re.search(r'\b[A-Za-z]+\s*\([^)]+\)\s*=', stripped))
    has_fraction    = '/' in stripped and '=' in stripped
    non_word        = sum(1 for c in stripped if not c.isalnum() and c not in ' .,;:\'"-')
    non_word_ratio  = non_word / max(len(stripped), 1)

    # --- Negative signal: prose with '=' ---
    # Example: "The encoder is composed of N = 6 identical layers."
    is_prose_with_eq = False
    if '=' in stripped:
        after_eq = stripped.split('=', 1)[1].strip()
        normal_words_after = [w for w in after_eq.split() if w.isalpha() and len(w) > 2]
        is_prose_with_eq = (
            len(normal_words_after) > 4
            and not has_greek
            and math_op_count == 0
            and not has_func_pattern
        )

    if is_prose_with_eq:
        return False

    return (
        has_greek
        or math_op_count >= 1
        or has_func_pattern
        or (has_fraction and non_word_ratio > 0.25)
        or (non_word_ratio > 0.35 and '=' in stripped)
    )


def detect_equations_in_chunk(text: str) -> Dict[str, bool]:
    """
    Analyze a chunk and return two flags:

    has_equation:      True if the chunk contains at least one equation line.
    equation_complete: True if the equation block starts AND ends inside
                       the chunk (not truncated across boundaries).

    Distinction matters for retrieval scoring:
    - has_equation=True,  equation_complete=True  → StructureBonus +0.15
    - has_equation=True,  equation_complete=False → StructureBonus -0.15 (truncated)
    """
    lines    = text.split('\n')
    eq_flags = [is_equation_line(l) for l in lines]
    has_equation = any(eq_flags)

    if not has_equation:
        return {"has_equation": False, "equation_complete": False}

    first_is_eq = eq_flags[0]
    last_is_eq  = eq_flags[-1]
    equation_complete = not first_is_eq and not last_is_eq

    return {"has_equation": True, "equation_complete": equation_complete}


# --- Quick validation ---
print("EQUATION DETECTION TEST")
print("=" * 55)
test_cases = [
    ("The encoder is composed of a stack of N = 6 identical layers.", False),
    ("Attention(Q, K, V) = softmax(QKᵀ / √dk) · V",                 True),
    ("α · Sim_sem(q,d) + β · Sim_lex(q,d)",                         True),
    ("layers, produce outputs of dimension dmodel = 512.",            False),
    ("Loss = -log exp(sim(q,d+)/τ) / Σ exp(sim(q,di)/τ)",           True),
]
all_correct = True
for text, expected in test_cases:
    result = is_equation_line(text)
    status = "✓" if result == expected else "✗ ERROR"
    if result != expected:
        all_correct = False
    label = "EQ" if result else "prose"
    print(f"  {status} [{label}] {text[:60]}")
print(f"\n{'✓ All cases correct' if all_correct else '✗ Errors found in detection'}")

EQUATION DETECTION TEST
  ✓ [prose] The encoder is composed of a stack of N = 6 identical layers
  ✓ [EQ] Attention(Q, K, V) = softmax(QKᵀ / √dk) · V
  ✓ [EQ] α · Sim_sem(q,d) + β · Sim_lex(q,d)
  ✓ [prose] layers, produce outputs of dimension dmodel = 512.
  ✓ [EQ] Loss = -log exp(sim(q,d+)/τ) / Σ exp(sim(q,di)/τ)

✓ All cases correct


# Strategy A — Small Fixed Chunking (100 tokens, 10% overlap)

In [4]:
def chunk_fixed_size(
    text: str,
    paper_id: str,
    chunk_size: int,
    overlap: float,
    strategy_name: str
) -> List[Dict[str, Any]]:
    """
    Split text into fixed-size chunks with overlap.

    Steps:
    1. Tokenize the full text into numeric IDs
    2. Create sliding windows of `chunk_size` tokens with stride `stride`
    3. Decode each window back to text
    4. Add metadata including equation detection

    Why tokenize first, then decode?
    BPE token boundaries don't align with word boundaries.
    Cutting by words could produce a "100-word" chunk with 130 tokens,
    silently exceeding model limits.
    """
    stride  = int(chunk_size * (1 - overlap))
    all_ids = TOKENIZER.encode(text, add_special_tokens=False)

    if len(all_ids) == 0:
        return []

    chunks      = []
    chunk_index = 0
    start       = 0

    while start < len(all_ids):
        end       = min(start + chunk_size, len(all_ids))
        token_ids = all_ids[start:end]

        chunk_text = TOKENIZER.decode(token_ids, skip_special_tokens=True).strip()

        if not chunk_text:
            start += stride
            continue

        eq_info   = detect_equations_in_chunk(chunk_text)
        is_last   = end >= len(all_ids)
        is_truncated = len(token_ids) < chunk_size and not is_last

        chunks.append({
            "chunk_id":          f"{paper_id}_{chunk_index:04d}",
            "paper_id":          paper_id,
            "chunk_index":       chunk_index,
            "text":              chunk_text,
            "token_count":       len(token_ids),
            "char_count":        len(chunk_text),
            "has_equation":      eq_info["has_equation"],
            "equation_complete": eq_info["equation_complete"],
            "is_truncated":      is_truncated,
            "strategy":          strategy_name,
        })

        chunk_index += 1
        start       += stride

    return chunks


# --- Generate Strategy A chunks ---
chunks_A = []

for txt_path in txt_files:
    paper_id = txt_path.stem
    text     = txt_path.read_text(encoding="utf-8")
    paper_chunks = chunk_fixed_size(
        text=text, paper_id=paper_id,
        chunk_size=100, overlap=0.10,
        strategy_name="small_fixed"
    )
    chunks_A.extend(paper_chunks)
    print(f"  {paper_id}: {len(paper_chunks)} chunks")

out_A = CHUNK / "strategy_A_small_fixed.json"
with open(out_A, "w", encoding="utf-8") as f:
    json.dump(chunks_A, f, indent=2, ensure_ascii=False)

print(f"\n✓ Strategy A: {len(chunks_A)} total chunks → {out_A}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (9869 > 512). Running this sequence through the model will result in indexing errors


  1502.04681v3: 110 chunks
  1512.03385v1: 176 chunks
  1910.13461v1: 90 chunks
  2312.11514v3: 231 chunks
  2312.15872v1: 81 chunks
  2312.17482v2: 216 chunks
  2407.15516v1: 89 chunks
  2504.08386v1: 88 chunks
  2606.03748v1: 355 chunks
  AttentionIsAllYouNeed: 79 chunks

✓ Strategy A: 1515 total chunks → ..\data\chunks\strategy_A_small_fixed.json


# Strategy B — Large Fixed Chunking (256 tokens, 10% overlap)


In [7]:
chunks_B = []

for txt_path in txt_files:
    paper_id = txt_path.stem
    text     = txt_path.read_text(encoding="utf-8")
    paper_chunks = chunk_fixed_size(
        text=text, paper_id=paper_id,
        chunk_size=256, overlap=0.10,
        strategy_name="large_fixed"
    )
    chunks_B.extend(paper_chunks)
    print(f"  {paper_id}: {len(paper_chunks)} chunks")

will_truncate = sum(1 for c in chunks_B if c["token_count"] > 256)
pct = will_truncate / len(chunks_B) * 100 if chunks_B else 0

out_B = CHUNK / "strategy_B_large_fixed.json"
with open(out_B, "w", encoding="utf-8") as f:
    json.dump(chunks_B, f, indent=2, ensure_ascii=False)

print(f"\n✓ Strategy B: {len(chunks_B)} total chunks → {out_B}")

  1502.04681v3: 43 chunks
  1512.03385v1: 69 chunks
  1910.13461v1: 35 chunks
  2312.11514v3: 91 chunks
  2312.15872v1: 32 chunks
  2312.17482v2: 85 chunks
  2407.15516v1: 35 chunks
  2504.08386v1: 35 chunks
  2606.03748v1: 139 chunks
  AttentionIsAllYouNeed: 31 chunks

✓ Strategy B: 595 total chunks → ..\data\chunks\strategy_B_large_fixed.json


# Strategy C — Semantic Chunking

In [10]:
MIN_TOKENS = 40
MAX_TOKENS = 256
def split_long_paragraph(text: str, max_tokens: int) -> List[str]:
    """
    Subdivide a long paragraph into fragments.
    Tries to split by sentences first, then falls back to newlines.
    """
    # First, try to split by sentences
    sentences = re.split(r'(?<=[.?!])\s+(?=[A-Z])', text)

    # If sentence splitting results in just one big chunk, it probably failed.
    # Fall back to splitting by newline characters. This is good for tables or lists.
    if len(sentences) == 1:
        sentences = text.split('\n')

    fragments = []
    current_sentences = []
    current_tokens    = 0

    for sent in sentences:
        # It's possible for a sentence to be empty if there are multiple newlines
        sent = sent.strip()
        if not sent:
            continue
            
        sent_tokens = count_tokens(sent)

        # If a single "sentence" (or line) is already too long, it must be split aggressively.
        # This is a hard fallback to prevent any chunk from ever exceeding the max.
        if sent_tokens > max_tokens:
            # If there's anything in the buffer, save it first
            if current_sentences:
                fragments.append(' '.join(current_sentences))
                current_sentences = []
                current_tokens = 0
            # Recursively split the oversized line
            fragments.extend(split_long_paragraph(sent, max_tokens))
            continue

        if current_tokens + sent_tokens > max_tokens and current_sentences:
            fragments.append(' '.join(current_sentences))
            current_sentences = [sent]
            current_tokens    = sent_tokens
        else:
            current_sentences.append(sent)
            current_tokens += sent_tokens

    if current_sentences:
        fragments.append(' '.join(current_sentences))

    return fragments


def chunk_semantic(
    text: str,
    paper_id: str,
    min_tokens: int = MIN_TOKENS,
    max_tokens: int = MAX_TOKENS
) -> List[Dict[str, Any]]:
    """
    Split text into semantic chunks respecting paragraph boundaries.

    Steps:
    1. Split by paragraphs (\\n\\n)
    2. Merge too-short paragraphs with the next one
    3. Split too-long paragraphs by sentences
    4. Build chunk objects with metadata
    """
    raw_paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]

    processed    = []
    buffer       = ""
    buffer_tokens = 0

    for para in raw_paragraphs:
        para_tokens = count_tokens(para)

        if para_tokens > max_tokens:
            if buffer:
                processed.append(buffer)
                buffer        = ""
                buffer_tokens = 0
            processed.extend(split_long_paragraph(para, max_tokens))

        elif para_tokens < min_tokens:
            buffer        = (buffer + ' ' + para).strip() if buffer else para
            buffer_tokens += para_tokens
            if buffer_tokens >= min_tokens:
                processed.append(buffer)
                buffer        = ""
                buffer_tokens = 0

        else:
            if buffer:
                combined        = buffer + ' ' + para
                combined_tokens = buffer_tokens + para_tokens
                if combined_tokens <= max_tokens:
                    processed.append(combined)
                else:
                    processed.append(buffer)
                    processed.append(para)
                buffer        = ""
                buffer_tokens = 0
            else:
                processed.append(para)

    if buffer:
        processed.append(buffer)

    chunks = []
    for i, chunk_text in enumerate(processed):
        chunk_text = chunk_text.strip()
        if not chunk_text:
            continue

        token_count = count_tokens(chunk_text)
        eq_info     = detect_equations_in_chunk(chunk_text)

        chunks.append({
            "chunk_id":          f"{paper_id}_{i:04d}",
            "paper_id":          paper_id,
            "chunk_index":       i,
            "text":              chunk_text,
            "token_count":       token_count,
            "char_count":        len(chunk_text),
            "has_equation":      eq_info["has_equation"],
            "equation_complete": eq_info["equation_complete"],
            "is_truncated":      False,   # semantic never truncates mid-token
            "strategy":          "semantic",
        })

    return chunks


# --- Generate Strategy C chunks ---
chunks_C = []

for txt_path in txt_files:
    paper_id = txt_path.stem
    text     = txt_path.read_text(encoding="utf-8")
    paper_chunks = chunk_semantic(text=text, paper_id=paper_id)
    chunks_C.extend(paper_chunks)
    print(f"  {paper_id}: {len(paper_chunks)} chunks")

out_C = CHUNK / "strategy_C_semantic.json"
with open(out_C, "w", encoding="utf-8") as f:
    json.dump(chunks_C, f, indent=2, ensure_ascii=False)

print(f"\n✓ Strategy C: {len(chunks_C)} total chunks → {out_C}")

  1502.04681v3: 46 chunks
  1512.03385v1: 85 chunks
  1910.13461v1: 41 chunks
  2312.11514v3: 129 chunks
  2312.15872v1: 46 chunks
  2312.17482v2: 110 chunks
  2407.15516v1: 42 chunks
  2504.08386v1: 44 chunks
  2606.03748v1: 205 chunks
  AttentionIsAllYouNeed: 43 chunks

✓ Strategy C: 791 total chunks → ..\data\chunks\strategy_C_semantic.json


# Analysis chunk

In [11]:
def analyze_strategy(chunks: List[Dict], name: str) -> Dict:
    """Compute descriptive statistics for a chunking strategy."""
    token_counts = [c["token_count"] for c in chunks]
    with_eq      = sum(1 for c in chunks if c["has_equation"])
    complete_eq  = sum(1 for c in chunks if c["equation_complete"])
    truncated    = sum(1 for c in chunks if c["is_truncated"])
    short_chunks = sum(1 for c in chunks if c["token_count"] < 30)

    return {
        "strategy":          name,
        "total_chunks":      len(chunks),
        "tokens_mean":       round(statistics.mean(token_counts), 1),
        "tokens_median":     round(statistics.median(token_counts), 1),
        "tokens_stdev":      round(statistics.stdev(token_counts), 1) if len(chunks) > 1 else 0,
        "tokens_min":        min(token_counts),
        "tokens_max":        max(token_counts),
        "pct_with_equation": round(with_eq / len(chunks) * 100, 1),
        "pct_eq_complete":   round(complete_eq / max(with_eq, 1) * 100, 1),
        "pct_truncated":     round(truncated / len(chunks) * 100, 1),
        "pct_short":         round(short_chunks / len(chunks) * 100, 1),
    }


results = [
    analyze_strategy(chunks_A, "A — Small Fixed (100 tok)"),
    analyze_strategy(chunks_B, "B — Large Fixed (300 tok)"),
    analyze_strategy(chunks_C, "C — Semantic"),
]

with open(META_DIR / "chunking_stats.json", "w") as f:
    json.dump(results, f, indent=2)

print("COMPARATIVE ANALYSIS OF CHUNKING STRATEGIES")
print("=" * 65)
print(f"{'Metric':<34} {'Config A':>9} {'Config B':>9} {'Config C':>9}")
print("-" * 65)

metrics = [
    ("Total chunks",                  "total_chunks"),
    ("Tokens: mean",                  "tokens_mean"),
    ("Tokens: median",                "tokens_median"),
    ("Tokens: std dev",               "tokens_stdev"),
    ("Tokens: min",                   "tokens_min"),
    ("Tokens: max",                   "tokens_max"),
    ("% chunks with equation",        "pct_with_equation"),
    ("% complete equations",          "pct_eq_complete"),
    ("% truncated chunks",            "pct_truncated"),
    ("% very short chunks (<30 tok)", "pct_short"),
]

for label, key in metrics:
    vals = [str(r[key]) for r in results]
    print(f"  {label:<32} {vals[0]:>9} {vals[1]:>9} {vals[2]:>9}")

print("\n" + "=" * 65)
a, b, c = results
ratio_ab = round(a['total_chunks'] / max(b['total_chunks'], 1), 1)
print(f"  Strategy A generates {ratio_ab}× more chunks than Strategy B.")
print(f"  Strategy C std dev: {c['tokens_stdev']} tokens "
      f"(vs {a['tokens_stdev']} in A, {b['tokens_stdev']} in B).")
if b['pct_truncated'] > 0:
    print(f"  ⚠  Strategy B: {b['pct_truncated']}% chunks will be truncated by MiniLM (>256 tokens).")
print(f"  Strategy C preserves complete equations best: {c['pct_eq_complete']}% "
      f"(A: {a['pct_eq_complete']}%, B: {b['pct_eq_complete']}%)")

COMPARATIVE ANALYSIS OF CHUNKING STRATEGIES
Metric                              Config A  Config B  Config C
-----------------------------------------------------------------
  Total chunks                          1515       595       791
  Tokens: mean                          99.6     253.9     171.7
  Tokens: median                         100       256       213
  Tokens: std dev                        5.4      18.5      82.4
  Tokens: min                              7        14         2
  Tokens: max                            100       256       256
  % chunks with equation                 1.5       0.0      18.1
  % complete equations                   0.0       0.0      83.9
  % truncated chunks                     0.0       0.0       0.0
  % very short chunks (<30 tok)          0.3       0.2       4.6

  Strategy A generates 2.5× more chunks than Strategy B.
  Strategy C std dev: 82.4 tokens (vs 5.4 in A, 18.5 in B).
  Strategy C preserves complete equations best: 83.9% (A: